# Phase 2 — Policy Comparison

**Run on Google Colab with GPU runtime.**

Goal: Compare all 7 gaze policies head-to-head on ImageNette and compute AUGC with paired bootstrap CIs.

Dataset: ImageNette val, 100-image fixed subset (seed=42). Development only — NOT official ImageNet-1k.

Policies compared:
- `random` — uniform random (seeded baseline)
- `center` — always fixates centre
- `coverage` — maximises distance from prior fixations
- `saliency_lowres` — spectral-residual saliency on 64px preview
- `saliency_ior` — saliency + inhibition-of-return penalty
- `inverse_saliency` — negative control (targets least salient)
- `ORACLE_saliency_fullres` — **ORACLE** (diagnostic only, never compared to valid policies)

Primary metric: AUGC (Area Under Accuracy-vs-Glimpses Curve) over T ∈ {0,1,2,3,4,5,6,8}.
Statistics: Paired bootstrap CI (10 000 samples) vs random baseline.

Creates:
- `canvit_results/phase2_raw.parquet`
- `canvit_results/phase2_policy_curves.png`
- `canvit_results/phase2_summary.csv`

## 2.0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['HF_HOME'] = '/content/drive/MyDrive/canvit_cache'
RESULTS_DIR = '/content/drive/MyDrive/canvit_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

REPO_URL = 'https://github.com/johnsaurabh/active-canvit-gaze.git'
REPO_DIR = '/content/active-canvit-gaze'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

!pip install -q 'canvit-pytorch @ git+https://github.com/m2b3/CanViT-PyTorch.git'
!pip install -q huggingface_hub scipy
print('Setup complete.')

## 2.1 — Load model

In [ ]:
import torch
import torch.nn.functional as F
from canvit_pytorch import CanViTForImageClassification
from canvit_pytorch import Viewpoint as CanViTViewpoint
from canvit_pytorch import sample_at_viewpoint
from canvit_pytorch.preprocess import preprocess

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CHECKPOINT = 'canvit/canvitb16-add-vpe-finetune-g128px-s512px-in1k-2026-04-06'
model = CanViTForImageClassification.from_pretrained(CHECKPOINT).eval().to(DEVICE)
print('Model loaded.')

CANVAS_GRID_SIZE = 32
GLIMPSE_SIZE_PX  = 128
SCENE_SIZE       = 512
LOCAL_SCALE      = 0.25
GLIMPSE_BUDGETS  = [0, 1, 2, 3, 4, 5, 6, 8]
N_LOCAL          = max(GLIMPSE_BUDGETS)  # 8

## 2.2 — Load ImageNette subset (same 100 images as Phase 1)

In [ ]:
import json, random, requests
from pathlib import Path
from PIL import Image

DATA_DIR       = '/content/drive/MyDrive/data'
IMAGENETTE_DIR = os.path.join(DATA_DIR, 'imagenette2-320')
VAL_DIR        = os.path.join(IMAGENETTE_DIR, 'val')

assert os.path.exists(VAL_DIR), f'ImageNette val not found at {VAL_DIR}. Run Phase 1 first.'

HEADERS = {'User-Agent': 'active-canvit-gaze/1.0'}
idx_url = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
class_idx = json.loads(requests.get(idx_url, headers=HEADERS, timeout=10).text)
synset_to_idx = {v[0]: int(k) for k, v in class_idx.items()}
idx_to_name   = {int(k): v[1] for k, v in class_idx.items()}
print(f'Loaded {len(synset_to_idx)} synset mappings.')

samples = []
for synset in sorted(os.listdir(VAL_DIR)):
    synset_dir = Path(VAL_DIR) / synset
    if not synset_dir.is_dir():
        continue
    label = synset_to_idx.get(synset, -1)
    for img_path in sorted(synset_dir.iterdir()):
        if img_path.suffix.lower() in {'.jpeg', '.jpg', '.png'}:
            samples.append({'path': str(img_path), 'synset': synset, 'label': label})

rng = random.Random(42)
subset = rng.sample(samples, min(100, len(samples)))
print(f'Subset: {len(subset)} images (seed=42) — identical to Phase 1.')

transform = preprocess(SCENE_SIZE)

## 2.3 — Instantiate policies

In [ ]:
from policies.random_policy    import RandomPolicy
from policies.center_policy    import CenterPolicy
from policies.coverage_policy  import CoveragePolicy
from policies.saliency_policy  import LowResSaliencyPolicy
from policies.saliency_ior_policy import SaliencyIORPolicy
from policies.negative_control import InverseSaliencyPolicy
from policies.oracle_policy    import FullResSaliencyOracle
from policies.base import PolicyType

# (name, policy instance, is_oracle)
POLICIES = [
    ('random',                RandomPolicy(scale=LOCAL_SCALE, seed=0)),
    ('center',                CenterPolicy(scale=LOCAL_SCALE)),
    ('coverage',              CoveragePolicy(scale=LOCAL_SCALE)),
    ('saliency_lowres',       LowResSaliencyPolicy(scale=LOCAL_SCALE)),
    ('saliency_ior',          SaliencyIORPolicy(scale=LOCAL_SCALE)),
    ('inverse_saliency',      InverseSaliencyPolicy(scale=LOCAL_SCALE)),
    ('ORACLE_saliency_fullres', FullResSaliencyOracle(scale=LOCAL_SCALE)),
]

print('Policies registered:')
for name, policy in POLICIES:
    m = policy.metadata
    tag = '[ORACLE]' if m.policy_type == PolicyType.ORACLE else '[valid ]'
    print(f'  {tag}  {name}')

## 2.4 — Policy runner

Runs T=0 full-scene glimpse, then T=1..8 policy-selected local glimpses.

Valid policies receive a 64px downsampled preview (`lowres_preview`).  
The ORACLE policy receives the full 512px image (`full_image`).  
Both are always passed; each policy uses only what it declares it needs.

In [ ]:
from policies.base import Viewpoint as PolicyViewpoint

LOWRES_SIZE = 64

def run_policy_sequence(policy, image_tensor, n_local=8):
    """
    image_tensor: [1, 3, 512, 512] on DEVICE
    Returns: (all_logits, viewpoint_history)
        all_logits: list of n_local+1 tensors, each [1, 1000] on CPU
        viewpoint_history: list of PolicyViewpoint objects
    """
    lowres = F.interpolate(
        image_tensor, size=(LOWRES_SIZE, LOWRES_SIZE),
        mode='bilinear', align_corners=False
    )  # [1, 3, 64, 64]

    policy.reset()
    state = model.init_state(batch_size=1, canvas_grid_size=CANVAS_GRID_SIZE)
    all_logits = []
    history = []  # list of PolicyViewpoint

    with torch.inference_mode():
        # T=0: full scene — same for every policy
        vp0 = PolicyViewpoint(x=0.0, y=0.0, s=1.0)
        centers, scales = vp0.to_canvit(DEVICE)
        cvp = CanViTViewpoint(centers=centers, scales=scales)
        glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=cvp, glimpse_size_px=GLIMPSE_SIZE_PX)
        logits, state = model(glimpse=glimpse, state=state, viewpoint=cvp)
        all_logits.append(logits.cpu())
        history.append(vp0)

        # T=1..n_local: policy-selected
        for _ in range(n_local):
            vp = policy.select_next(
                viewpoint_history=history,
                model_state=state,
                lowres_preview=lowres,
                full_image=image_tensor,
            )
            centers, scales = vp.to_canvit(DEVICE)
            cvp = CanViTViewpoint(centers=centers, scales=scales)
            glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=cvp, glimpse_size_px=GLIMPSE_SIZE_PX)
            logits, state = model(glimpse=glimpse, state=state, viewpoint=cvp)
            all_logits.append(logits.cpu())
            history.append(vp)

    return all_logits, history

## 2.5 — Run all policies

7 policies × 100 images × 9 timesteps. Expect ~15–25 min on Colab T4.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

all_records = []
# per_policy_correct[policy_name] = (N_images, N_budgets) bool array
per_policy_correct = {}
# per_policy_viewpoints[policy_name] = list of viewpoint sequences (one per image)
per_policy_viewpoints = {}

for policy_name, policy in POLICIES:
    print(f'\n--- {policy_name} ---')
    correct_matrix = []  # (N_images, N_budgets) at GLIMPSE_BUDGETS steps
    vp_sequences = []

    for sample in tqdm(subset, desc=policy_name, leave=False):
        try:
            img = Image.open(sample['path']).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)
            all_logits, history = run_policy_sequence(policy, img_tensor, n_local=N_LOCAL)

            correct_at_budgets = []
            for t, logits in enumerate(all_logits):
                if t not in GLIMPSE_BUDGETS:
                    continue
                probs = torch.softmax(logits, dim=-1)
                pred  = int(probs.argmax())
                conf  = float(probs.max())
                top5  = torch.topk(probs, k=5, dim=-1).indices[0].tolist()
                correct_top1 = int(pred == sample['label'])
                correct_top5 = int(sample['label'] in top5)
                correct_at_budgets.append(correct_top1)

                all_records.append({
                    'policy':       policy_name,
                    'image_id':     sample['path'],
                    'synset':       sample['synset'],
                    'true_label':   sample['label'],
                    'timestep':     t,
                    'pred_label':   pred,
                    'confidence':   conf,
                    'correct_top1': correct_top1,
                    'correct_top5': correct_top5,
                })

            correct_matrix.append(correct_at_budgets)
            vp_sequences.append([(vp.x, vp.y) for vp in history])

        except Exception as e:
            print(f'  FAIL: {sample["path"]}: {e}')

    per_policy_correct[policy_name]    = np.array(correct_matrix, dtype=np.float32)
    per_policy_viewpoints[policy_name] = vp_sequences
    mean_t0  = per_policy_correct[policy_name][:, 0].mean()
    mean_t8  = per_policy_correct[policy_name][:, -1].mean()
    print(f'  T=0: {mean_t0:.3f}   T=8: {mean_t8:.3f}')

df = pd.DataFrame(all_records)
print(f'\nTotal records: {len(df)}')

## 2.6 — Compute AUGC

In [ ]:
from evaluation.metrics import compute_accuracy_curve, paired_bootstrap_ci, compute_spatial_metrics

augc_results = {}  # policy_name -> (mean_accuracy_curve, per_image_augc)

for policy_name, correct_matrix in per_policy_correct.items():
    mean_acc, per_img_augc = compute_accuracy_curve(correct_matrix, GLIMPSE_BUDGETS)
    augc_results[policy_name] = (mean_acc, per_img_augc)

print('Mean AUGC per policy:')
print(f'{"Policy":<28} {"AUGC":>8}')
print('-' * 38)
for policy_name, (_, per_img_augc) in augc_results.items():
    tag = ' [ORACLE]' if 'ORACLE' in policy_name else ''
    print(f'{policy_name:<28} {per_img_augc.mean():>8.4f}{tag}')

## 2.7 — Accuracy-vs-Glimpses plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Separate valid policies from oracle for display
VALID_POLICIES   = [(n, p) for n, p in POLICIES if 'ORACLE' not in n]
ORACLE_POLICIES  = [(n, p) for n, p in POLICIES if 'ORACLE' in n]

COLORS = {
    'random':                 '#888888',
    'center':                 '#4e79a7',
    'coverage':               '#f28e2b',
    'saliency_lowres':        '#59a14f',
    'saliency_ior':           '#e15759',
    'inverse_saliency':       '#b07aa1',
    'ORACLE_saliency_fullres': '#76b7b2',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: valid policies only
ax = axes[0]
for name, _ in VALID_POLICIES:
    mean_acc, _ = augc_results[name]
    ls = '--' if name == 'inverse_saliency' else '-'
    ax.plot(GLIMPSE_BUDGETS, mean_acc, ls, color=COLORS[name], marker='o', ms=4, label=name)
ax.set_xlabel('Glimpse budget')
ax.set_ylabel('Top-1 accuracy')
ax.set_title('Valid policies — Accuracy vs Glimpses\n(ImageNette dev, 100 images)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Right: all policies including oracle (oracle dashed, thinner)
ax = axes[1]
for name, _ in VALID_POLICIES:
    mean_acc, _ = augc_results[name]
    ls = '--' if name == 'inverse_saliency' else '-'
    ax.plot(GLIMPSE_BUDGETS, mean_acc, ls, color=COLORS[name], marker='o', ms=4, label=name)
for name, _ in ORACLE_POLICIES:
    mean_acc, _ = augc_results[name]
    ax.plot(GLIMPSE_BUDGETS, mean_acc, ':', color=COLORS[name], marker='s', ms=4,
            linewidth=1.5, label=f'{name} [ORACLE — not comparable]')
ax.set_xlabel('Glimpse budget')
ax.set_ylabel('Top-1 accuracy')
ax.set_title('All policies (oracle dotted — diagnostic only)\n(ImageNette dev, 100 images)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'phase2_policy_curves.png')
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

## 2.8 — Statistical comparison vs random

Paired bootstrap CI on per-image AUGC. Valid active policies only.

In [ ]:
_, random_augc = augc_results['random']

print('Paired bootstrap CI (10 000 samples) — AUGC diff vs random baseline')
print('Valid active policies only. Oracle excluded from this comparison.')
print()
print(f'{"Policy":<28} {"AUGC":>8}  {"Diff vs random":>16}  {"95% CI":>22}  {"Beats random?"}') 
print('-' * 90)

summary_rows = []
for name, _ in VALID_POLICIES:
    _, per_img_augc = augc_results[name]
    mean_augc = float(per_img_augc.mean())

    diff, lo, hi = paired_bootstrap_ci(per_img_augc, random_augc, n_bootstrap=10_000)
    beats = 'YES' if lo > 0 else ('NO' if hi < 0 else 'uncertain')

    print(f'{name:<28} {mean_augc:>8.4f}  {diff:>+16.4f}  [{lo:>+8.4f}, {hi:>+8.4f}]  {beats}')
    summary_rows.append({
        'policy':       name,
        'mean_augc':    mean_augc,
        'diff_vs_random': diff,
        'ci_lower':     lo,
        'ci_upper':     hi,
        'beats_random': beats,
        'type':         'valid',
    })

print()
print('ORACLE (diagnostic only — not compared to valid policies):')
for name, _ in ORACLE_POLICIES:
    _, per_img_augc = augc_results[name]
    print(f'  {name}: AUGC = {per_img_augc.mean():.4f}')
    summary_rows.append({
        'policy':       name,
        'mean_augc':    float(per_img_augc.mean()),
        'diff_vs_random': float('nan'),
        'ci_lower':     float('nan'),
        'ci_upper':     float('nan'),
        'beats_random': 'N/A (ORACLE)',
        'type':         'oracle',
    })

summary_df = pd.DataFrame(summary_rows)

## 2.9 — Spatial behaviour metrics

In [ ]:
print('Spatial behaviour metrics (local fixations only, T=1..8):')
print()
print(f'{"Policy":<28} {"Mean displ":>12}  {"Revisit rate":>14}  {"Mean |center|":>14}')
print('-' * 72)

for name, _ in POLICIES:
    seqs = per_policy_viewpoints[name]
    metrics = compute_spatial_metrics(seqs)
    tag = ' [ORACLE]' if 'ORACLE' in name else ''
    print(
        f'{name:<28} '
        f'{metrics.get("mean_displacement", float("nan")):>12.4f}  '
        f'{metrics.get("revisit_rate", float("nan")):>14.4f}  '
        f'{metrics.get("mean_center_distance", float("nan")):>14.4f}{tag}'
    )

## 2.10 — Save results

In [ ]:
raw_path     = os.path.join(RESULTS_DIR, 'phase2_raw.parquet')
summary_path = os.path.join(RESULTS_DIR, 'phase2_summary.csv')

df.to_parquet(raw_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f'Raw records:  {raw_path}')
print(f'Summary:      {summary_path}')

## 2.11 — Phase 2 verdict

In [ ]:
print('=' * 60)
print('PHASE 2 POLICY COMPARISON — SUMMARY')
print('=' * 60)
print(f'Dataset:   ImageNette val, {len(subset)}-image subset (seed=42). Dev only.')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Budgets:   {GLIMPSE_BUDGETS}')
print(f'Metric:    AUGC (Area Under Accuracy-vs-Glimpses Curve)')
print()

# Rank valid policies by AUGC
ranked = sorted(
    [(n, float(augc_results[n][1].mean())) for n, _ in VALID_POLICIES],
    key=lambda x: -x[1]
)

print('Valid policy ranking (by AUGC):')
for rank, (name, augc) in enumerate(ranked, 1):
    row = summary_df[summary_df['policy'] == name].iloc[0]
    ci  = f"[{row['ci_lower']:+.4f}, {row['ci_upper']:+.4f}]"
    print(f'  {rank}. {name:<28} AUGC={augc:.4f}  diff_vs_random={row["diff_vs_random"]:+.4f}  95%CI={ci}  {row["beats_random"]}')

print()
print('ORACLE (not in ranking):')
for name, _ in ORACLE_POLICIES:
    augc = float(augc_results[name][1].mean())
    print(f'  {name}: AUGC={augc:.4f} [diagnostic upper bound]')

print()
print('Interpretation note: ImageNette N=100 is development-only.')
print('CIs will be wide. Treat as directional signal, not final result.')
print()
print('Next: log results in docs/experiment_log.md as EXP-002,')
print('      then discuss findings before writing the report.')